# Analysis of Marker Embeddings Evolution (stream_id_A and stream_id_B)

This notebook analyzes how marker embeddings (stream_id_A and stream_id_B) change during training.

**Goal**: Track the dynamics of marker embeddings changes across saved checkpoints.


In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from tqdm.auto import tqdm
import pandas as pd

# Plot setup
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Increase default plot size
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12


## 1. Loading Configuration and Checkpoint List


In [4]:
# Path to the checkpoint directory
RUN_DIR = Path('/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251221_112940')

# Loading configuration
with open(RUN_DIR / 'training_config.json', 'r') as f:
    config = json.load(f)

print("Training Configuration:")
print("=" * 80)
for key, value in config.items():
    print(f"{key}: {value}")
print("=" * 80)


Training Configuration:
model_name: EleutherAI/pythia-2.8b
dataset_name: HuggingFaceFW/fineweb
dataset_config: None
dataset_split: train
device: cuda:0
batch_size: 8
learning_rate: 0.001
backbone_lr: 0.0001
freeze_backbone: False
num_train_steps: 200000
max_length: 1024
log_interval: 1
similarity_log_interval: 50
eval_interval: 500
save_interval: 5000
max_gen_tokens: 50
temperature: 0.7
seed: 42
objective: separate
use_loss_balancing: True
shared_embeddings: True
embedding_init_strategy: copy
unembedding_init_strategy: None
embedding_init_noise_std: 0.01
mixup_alpha: 0.4
same_text_prob: 0.2
teacher_model_name: None
teacher_load_in_8bit: True
distillation_temperature: 2.0
loss_balance_strength: 5.0
loss_balance_ema_momentum: 0.99
loss_balance_warmup_steps: 100
loss_balance_clip_min: 0.1
loss_balance_clip_max: 10.0
lr_scheduler_patience: 10
lr_scheduler_factor: 0.5
gradient_accumulation_steps: 2
world_size: 1


In [5]:
# Find all checkpoints
checkpoint_dirs = sorted(
    [d for d in RUN_DIR.iterdir() if d.is_dir() and d.name.startswith('checkpoint-')],
    key=lambda x: int(x.name.split('-')[1])
 )

checkpoint_steps = [int(d.name.split('-')[1]) for d in checkpoint_dirs]

print(f"\nFound checkpoints: {len(checkpoint_dirs)}")
print(f"Steps: {checkpoint_steps}")



Found checkpoints: 12
Steps: [10000, 20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000, 100000, 110000, 120000]


## 2. Extracting Marker Embeddings from Checkpoints


In [26]:
def load_marker_embeddings(checkpoint_path):
    """
    Loads marker embeddings stream_id_A and stream_id_B from checkpoint.
    
    Returns:
        tuple: (stream_id_A, stream_id_B) as numpy arrays
    """
    checkpoint = torch.load(checkpoint_path / 'model.pt', map_location='cpu')
    
    # Extract marker embeddings from model_state_dict
    # They have shape [1, 1, hidden_dim]
    stream_id_A = checkpoint['model_state_dict']['stream_id_A'].squeeze().float().numpy()  # [hidden_dim]
    stream_id_B = checkpoint['model_state_dict']['stream_id_B'].squeeze().float().numpy()  # [hidden_dim]
    
    return stream_id_A, stream_id_B


In [27]:
# Load marker vectors from all checkpoints
markers_A = []
markers_B = []

print("Loading marker embeddings from checkpoints...")

for checkpoint_dir in tqdm(checkpoint_dirs):
    stream_id_A, stream_id_B = load_marker_embeddings(checkpoint_dir)
    markers_A.append(stream_id_A)
    markers_B.append(stream_id_B)

# Convert to numpy arrays for convenience
# Shape: [num_checkpoints, hidden_dim]
markers_A = np.array(markers_A)
markers_B = np.array(markers_B)

print(f"\nShape of markers_A: {markers_A.shape}")
print(f"Shape of markers_B: {markers_B.shape}")


Loading marker embeddings from checkpoints...


  0%|          | 0/12 [00:00<?, ?it/s]

/tmp/ipykernel_632585/2604142106.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path / 'model.pt', map_location='cpu')


RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory

## 3. Analysis of Marker Evolution Metrics


In [ ]:
def compute_metrics(markers_A, markers_B):
    """
    Computes various metrics for marker embeddings.
    
    Returns:
        dict: dictionary with metrics
    """
    num_checkpoints = len(markers_A)
    
    metrics = {
        'norm_A': [],
        'norm_B': [],
        'cosine_sim_AB': [],  # cosine similarity between A and B
        'euclidean_dist_AB': [],  # euclidean distance between A and B
        'norm_diff_AB': [],  # norm difference
        'mean_A': [],  # mean component value
        'mean_B': [],
        'std_A': [],  # component standard deviation
        'std_B': [],
    }
    
    for i in range(num_checkpoints):
        marker_A = markers_A[i]
        marker_B = markers_B[i]
        
        # Norms
        norm_A = np.linalg.norm(marker_A)
        norm_B = np.linalg.norm(marker_B)
        metrics['norm_A'].append(norm_A)
        metrics['norm_B'].append(norm_B)
        
        # Cosine similarity
        cosine_sim = np.dot(marker_A, marker_B) / (norm_A * norm_B + 1e-8)
        metrics['cosine_sim_AB'].append(cosine_sim)
        
        # Euclidean distance
        euclidean_dist = np.linalg.norm(marker_A - marker_B)
        metrics['euclidean_dist_AB'].append(euclidean_dist)
        
        # Norm difference
        norm_diff = abs(norm_A - norm_B)
        metrics['norm_diff_AB'].append(norm_diff)
        
        # Component statistics
        metrics['mean_A'].append(np.mean(marker_A))
        metrics['mean_B'].append(np.mean(marker_B))
        metrics['std_A'].append(np.std(marker_A))
        metrics['std_B'].append(np.std(marker_B))
    
    return metrics

# Compute metrics
metrics = compute_metrics(markers_A, markers_B)

# Create DataFrame for convenience
df_metrics = pd.DataFrame(metrics)
df_metrics['step'] = checkpoint_steps
df_metrics = df_metrics.set_index('step')

print("Metrics computed!")
df_metrics.head()


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df_metrics.index, df_metrics['norm_A'], label='Norm(stream_id_A)', marker='o', linewidth=2)
ax.plot(df_metrics.index, df_metrics['norm_B'], label='Norm(stream_id_B)', marker='s', linewidth=2)

ax.set_xlabel('Training Step', fontsize=14)
ax.set_ylabel('L2 Norm', fontsize=14)
ax.set_title('Evolution of Marker Embedding Norms', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics of norm changes
print("\nNorm statistics:")
print(f"stream_id_A - start: {df_metrics['norm_A'].iloc[0]:.4f}, end: {df_metrics['norm_A'].iloc[-1]:.4f}, change: {df_metrics['norm_A'].iloc[-1] - df_metrics['norm_A'].iloc[0]:.4f}")
print(f"stream_id_B - start: {df_metrics['norm_B'].iloc[0]:.4f}, end: {df_metrics['norm_B'].iloc[-1]:.4f}, change: {df_metrics['norm_B'].iloc[-1] - df_metrics['norm_B'].iloc[0]:.4f}")


## 5. Analysis of Cosine Similarity Between Markers


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df_metrics.index, df_metrics['cosine_sim_AB'], label='Cosine Similarity', 
        marker='o', linewidth=2, color='#e74c3c')

ax.set_xlabel('Training Step', fontsize=14)
ax.set_ylabel('Cosine Similarity', fontsize=14)
ax.set_title('Cosine Similarity between stream_id_A and stream_id_B', fontsize=16, fontweight='bold')
ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics
print("\nCosine similarity statistics:")
print(f"Start: {df_metrics['cosine_sim_AB'].iloc[0]:.4f}")
print(f"End: {df_metrics['cosine_sim_AB'].iloc[-1]:.4f}")
print(f"Change: {df_metrics['cosine_sim_AB'].iloc[-1] - df_metrics['cosine_sim_AB'].iloc[0]:.4f}")
print(f"Minimum: {df_metrics['cosine_sim_AB'].min():.4f} (step {df_metrics['cosine_sim_AB'].idxmin()})")
print(f"Maximum: {df_metrics['cosine_sim_AB'].max():.4f} (step {df_metrics['cosine_sim_AB'].idxmax()})")


## 6. Euclidean Distance Between Markers


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df_metrics.index, df_metrics['euclidean_dist_AB'], label='Euclidean Distance', 
        marker='o', linewidth=2, color='#9b59b6')

ax.set_xlabel('Training Step', fontsize=14)
ax.set_ylabel('Euclidean Distance', fontsize=14)
ax.set_title('Euclidean Distance between stream_id_A and stream_id_B', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics
print("\nEuclidean distance statistics:")
print(f"Start: {df_metrics['euclidean_dist_AB'].iloc[0]:.4f}")
print(f"End: {df_metrics['euclidean_dist_AB'].iloc[-1]:.4f}")
print(f"Change: {df_metrics['euclidean_dist_AB'].iloc[-1] - df_metrics['euclidean_dist_AB'].iloc[0]:.4f}")


## 7. Marker Embedding Component Statistics


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Mean components
axes[0, 0].plot(df_metrics.index, df_metrics['mean_A'], label='Mean(stream_id_A)', marker='o', linewidth=2)
axes[0, 0].plot(df_metrics.index, df_metrics['mean_B'], label='Mean(stream_id_B)', marker='s', linewidth=2)
axes[0, 0].set_xlabel('Training Step')
axes[0, 0].set_ylabel('Mean Value')
axes[0, 0].set_title('Mean Component Values')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axhline(y=0, color='k', linestyle='--', alpha=0.3)

# Std components
axes[0, 1].plot(df_metrics.index, df_metrics['std_A'], label='Std(stream_id_A)', marker='o', linewidth=2)
axes[0, 1].plot(df_metrics.index, df_metrics['std_B'], label='Std(stream_id_B)', marker='s', linewidth=2)
axes[0, 1].set_xlabel('Training Step')
axes[0, 1].set_ylabel('Standard Deviation')
axes[0, 1].set_title('Component Standard Deviation')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Norm difference
axes[1, 0].plot(df_metrics.index, df_metrics['norm_diff_AB'], label='|Norm(A) - Norm(B)|', 
                marker='o', linewidth=2, color='#f39c12')
axes[1, 0].set_xlabel('Training Step')
axes[1, 0].set_ylabel('Norm Difference')
axes[1, 0].set_title('Norm Difference Between Markers')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Summary info - trajectory in space (Norm A vs Norm B)
axes[1, 1].scatter(df_metrics['norm_A'], df_metrics['norm_B'], 
                   c=df_metrics.index, cmap='viridis', s=100, alpha=0.7)
axes[1, 1].plot(df_metrics['norm_A'], df_metrics['norm_B'], 
                alpha=0.3, linestyle='--', color='gray')
axes[1, 1].set_xlabel('Norm(stream_id_A)')
axes[1, 1].set_ylabel('Norm(stream_id_B)')
axes[1, 1].set_title('Norm Trajectory in Space')
axes[1, 1].grid(True, alpha=0.3)
# Add diagonal for reference
max_norm = max(df_metrics['norm_A'].max(), df_metrics['norm_B'].max())
min_norm = min(df_metrics['norm_A'].min(), df_metrics['norm_B'].min())
axes[1, 1].plot([min_norm, max_norm], [min_norm, max_norm], 'k--', alpha=0.3, label='y=x')
axes[1, 1].legend()

# Colorbar for the last plot
cbar = plt.colorbar(axes[1, 1].collections[0], ax=axes[1, 1])
cbar.set_label('Training Step', rotation=270, labelpad=20)

plt.tight_layout()
plt.show()


In [ ]:
# Compute change of each component from first to last checkpoint
delta_A = markers_A[-1] - markers_A[0]
delta_B = markers_B[-1] - markers_B[0]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Distribution of component changes for stream_id_A
axes[0, 0].hist(delta_A, bins=50, alpha=0.7, color='#3498db', edgecolor='black')
axes[0, 0].set_xlabel('Δ Component Value')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Component Changes (stream_id_A)')
axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].grid(True, alpha=0.3)

# Distribution of component changes for stream_id_B
axes[0, 1].hist(delta_B, bins=50, alpha=0.7, color='#2ecc71', edgecolor='black')
axes[0, 1].set_xlabel('Δ Component Value')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Component Changes (stream_id_B)')
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 1].grid(True, alpha=0.3)

# Top-10 components with largest change (stream_id_A)
top_indices_A = np.argsort(np.abs(delta_A))[-10:]
axes[1, 0].barh(range(10), delta_A[top_indices_A], color='#3498db')
axes[1, 0].set_yticks(range(10))
axes[1, 0].set_yticklabels([f'Comp {i}' for i in top_indices_A])
axes[1, 0].set_xlabel('Δ Value')
axes[1, 0].set_title('Top-10 Components with Largest Change (stream_id_A)')
axes[1, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1, 0].grid(True, alpha=0.3, axis='x')

# Top-10 components with largest change (stream_id_B)
top_indices_B = np.argsort(np.abs(delta_B))[-10:]
axes[1, 1].barh(range(10), delta_B[top_indices_B], color='#2ecc71')
axes[1, 1].set_yticks(range(10))
axes[1, 1].set_yticklabels([f'Comp {i}' for i in top_indices_B])
axes[1, 1].set_xlabel('Δ Value')
axes[1, 1].set_title('Top-10 Components with Largest Change (stream_id_B)')
axes[1, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"\nComponent change statistics:")
print(f"stream_id_A:")
print(f"  Mean change: {np.mean(delta_A):.6f}")
print(f"  Std of change: {np.std(delta_A):.6f}")
print(f"  Max increase: {np.max(delta_A):.6f}")
print(f"  Max decrease: {np.min(delta_A):.6f}")
print(f"\nstream_id_B:")
print(f"  Mean change: {np.mean(delta_B):.6f}")
print(f"  Std of change: {np.std(delta_B):.6f}")
print(f"  Max increase: {np.max(delta_B):.6f}")
print(f"  Max decrease: {np.min(delta_B):.6f}")


In [ ]:
# Select 50 components with highest temporal variation
def select_most_varying_components(markers, n_components=50):
    """Selects n components with highest temporal variation."""
    variances = np.var(markers, axis=0)
    top_indices = np.argsort(variances)[-n_components:]
    return top_indices

# Select components
top_varying_A = select_most_varying_components(markers_A, n_components=50)
top_varying_B = select_most_varying_components(markers_B, n_components=50)

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# Heatmap for stream_id_A
im1 = axes[0].imshow(markers_A[:, top_varying_A].T, aspect='auto', cmap='RdBu_r', 
                     interpolation='nearest')
axes[0].set_xlabel('Checkpoint Index', fontsize=12)
axes[0].set_ylabel('Component Index', fontsize=12)
axes[0].set_title('Evolution of Top-50 Varying Components (stream_id_A)', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(len(checkpoint_steps)))
axes[0].set_xticklabels([f'{step//1000}K' for step in checkpoint_steps], rotation=45)
plt.colorbar(im1, ax=axes[0], label='Component Value')

# Heatmap for stream_id_B
im2 = axes[1].imshow(markers_B[:, top_varying_B].T, aspect='auto', cmap='RdBu_r', 
                     interpolation='nearest')
axes[1].set_xlabel('Checkpoint Index', fontsize=12)
axes[1].set_ylabel('Component Index', fontsize=12)
axes[1].set_title('Evolution of Top-50 Varying Components (stream_id_B)', fontsize=14, fontweight='bold')
axes[1].set_xticks(range(len(checkpoint_steps)))
axes[1].set_xticklabels([f'{step//1000}K' for step in checkpoint_steps], rotation=45)
plt.colorbar(im2, ax=axes[1], label='Component Value')

plt.tight_layout()
plt.show()


## 10. Analysis of Marker Change Rate

We will calculate and visualize the rate of change of marker vectors between consecutive checkpoints.


In [ ]:
# Calculate the difference between consecutive checkpoints
deltas_A = np.diff(markers_A, axis=0)
deltas_B = np.diff(markers_B, axis=0)

# Calculate the norm of changes (change rate)
change_rate_A = np.linalg.norm(deltas_A, axis=1)
change_rate_B = np.linalg.norm(deltas_B, axis=1)

# Midpoints of intervals for the plot
step_midpoints = [(checkpoint_steps[i] + checkpoint_steps[i+1]) / 2 
                  for i in range(len(checkpoint_steps) - 1)]

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Change rate
axes[0].plot(step_midpoints, change_rate_A, label='Change Rate (stream_id_A)', 
             marker='o', linewidth=2, color='#3498db')
axes[0].plot(step_midpoints, change_rate_B, label='Change Rate (stream_id_B)', 
             marker='s', linewidth=2, color='#2ecc71')
axes[0].set_xlabel('Training Step', fontsize=12)
axes[0].set_ylabel('L2 Norm of Change', fontsize=12)
axes[0].set_title('Rate of Change of Marker Vectors Between Checkpoints', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Logarithmic scale for better visualization
axes[1].semilogy(step_midpoints, change_rate_A, label='Change Rate (stream_id_A)', 
                 marker='o', linewidth=2, color='#3498db')
axes[1].semilogy(step_midpoints, change_rate_B, label='Change Rate (stream_id_B)', 
                 marker='s', linewidth=2, color='#2ecc71')
axes[1].set_xlabel('Training Step', fontsize=12)
axes[1].set_ylabel('L2 Norm of Change (log scale)', fontsize=12)
axes[1].set_title('Rate of Change (logarithmic scale)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nChange rate statistics:")
print(f"stream_id_A:")
print(f"  Mean rate: {np.mean(change_rate_A):.6f}")
print(f"  Rate Std: {np.std(change_rate_A):.6f}")
print(f"  Max rate: {np.max(change_rate_A):.6f} (between steps {checkpoint_steps[np.argmax(change_rate_A)]} and {checkpoint_steps[np.argmax(change_rate_A)+1]})")
print(f"  Min rate: {np.min(change_rate_A):.6f} (between steps {checkpoint_steps[np.argmin(change_rate_A)]} and {checkpoint_steps[np.argmin(change_rate_A)+1]})")
print(f"\nstream_id_B:")
print(f"  Mean rate: {np.mean(change_rate_B):.6f}")
print(f"  Rate Std: {np.std(change_rate_B):.6f}")
print(f"  Max rate: {np.max(change_rate_B):.6f} (between steps {checkpoint_steps[np.argmax(change_rate_B)]} and {checkpoint_steps[np.argmax(change_rate_B)+1]})")
print(f"  Min rate: {np.min(change_rate_B):.6f} (between steps {checkpoint_steps[np.argmin(change_rate_B)]} and {checkpoint_steps[np.argmin(change_rate_B)+1]})")


## 11. PCA Analysis of Marker Trajectory

We will analyze the trajectory of markers in the principal component space.


In [ ]:
from sklearn.decomposition import PCA

# Combine markers for joint PCA
markers_combined = np.vstack([markers_A, markers_B])

# Apply PCA
pca = PCA(n_components=2)
markers_pca = pca.fit_transform(markers_combined)

# Split back into A and B
markers_A_pca = markers_pca[:len(markers_A)]
markers_B_pca = markers_pca[len(markers_A):]

fig, ax = plt.subplots(figsize=(12, 10))

# Trajectory stream_id_A
scatter_A = ax.scatter(markers_A_pca[:, 0], markers_A_pca[:, 1], 
                       c=checkpoint_steps, cmap='Blues', s=150, alpha=0.7,
                       marker='o', edgecolors='black', linewidths=1.5, label='stream_id_A')
ax.plot(markers_A_pca[:, 0], markers_A_pca[:, 1], 
        alpha=0.3, linestyle='-', color='blue', linewidth=2)

# Trajectory stream_id_B
scatter_B = ax.scatter(markers_B_pca[:, 0], markers_B_pca[:, 1], 
                       c=checkpoint_steps, cmap='Greens', s=150, alpha=0.7,
                       marker='s', edgecolors='black', linewidths=1.5, label='stream_id_B')
ax.plot(markers_B_pca[:, 0], markers_B_pca[:, 1], 
        alpha=0.3, linestyle='-', color='green', linewidth=2)

# Mark start and end points
ax.scatter(markers_A_pca[0, 0], markers_A_pca[0, 1], 
           s=300, marker='*', color='red', edgecolors='black', linewidths=2, 
           label='Start A', zorder=5)
ax.scatter(markers_A_pca[-1, 0], markers_A_pca[-1, 1], 
           s=300, marker='P', color='darkred', edgecolors='black', linewidths=2, 
           label='End A', zorder=5)
ax.scatter(markers_B_pca[0, 0], markers_B_pca[0, 1], 
           s=300, marker='*', color='lime', edgecolors='black', linewidths=2, 
           label='Start B', zorder=5)
ax.scatter(markers_B_pca[-1, 0], markers_B_pca[-1, 1], 
           s=300, marker='P', color='darkgreen', edgecolors='black', linewidths=2, 
           label='End B', zorder=5)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}% variance)', fontsize=14)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}% variance)', fontsize=14)
ax.set_title('Trajectory of Marker Vectors in Principal Component Space', 
             fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nExplained variance of first 2 PCs: {pca.explained_variance_ratio_[:2].sum()*100:.2f}%")
print(f"PC1: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"PC2: {pca.explained_variance_ratio_[1]*100:.2f}%")


In [ ]:
# Create a summary table with key metrics
summary = {
    'Metric': [
        'Norm(stream_id_A) - start',
        'Norm(stream_id_A) - end',
        'Norm(stream_id_A) - change',
        'Norm(stream_id_B) - start',
        'Norm(stream_id_B) - end',
        'Norm(stream_id_B) - change',
        'Cosine Similarity - start',
        'Cosine Similarity - end',
        'Cosine Similarity - change',
        'Euclidean Distance - start',
        'Euclidean Distance - end',
        'Euclidean Distance - change',
    ],
    'Value': [
        df_metrics['norm_A'].iloc[0],
        df_metrics['norm_A'].iloc[-1],
        df_metrics['norm_A'].iloc[-1] - df_metrics['norm_A'].iloc[0],
        df_metrics['norm_B'].iloc[0],
        df_metrics['norm_B'].iloc[-1],
        df_metrics['norm_B'].iloc[-1] - df_metrics['norm_B'].iloc[0],
        df_metrics['cosine_sim_AB'].iloc[0],
        df_metrics['cosine_sim_AB'].iloc[-1],
        df_metrics['cosine_sim_AB'].iloc[-1] - df_metrics['cosine_sim_AB'].iloc[0],
        df_metrics['euclidean_dist_AB'].iloc[0],
        df_metrics['euclidean_dist_AB'].iloc[-1],
        df_metrics['euclidean_dist_AB'].iloc[-1] - df_metrics['euclidean_dist_AB'].iloc[0],
    ]
}

df_summary = pd.DataFrame(summary)
df_summary['Value'] = df_summary['Value'].apply(lambda x: f'{x:.6f}')

print("\n" + "="*80)
print("SUMMARY METRICS TABLE")
print("="*80)
print(df_summary.to_string(index=False))
print("="*80)


## 13. Data Export for Further Analysis


In [ ]:
# Save all data for further analysis
output_dir = RUN_DIR / 'marker_analysis'
output_dir.mkdir(exist_ok=True)

# Save metrics
df_metrics.to_csv(output_dir / 'marker_metrics.csv')
print(f"Metrics saved to: {output_dir / 'marker_metrics.csv'}")

# Save raw marker data
np.save(output_dir / 'markers_A.npy', markers_A)
np.save(output_dir / 'markers_B.npy', markers_B)
print(f"Raw marker data saved to: {output_dir}")

# Save checkpoint_steps
np.save(output_dir / 'checkpoint_steps.npy', np.array(checkpoint_steps))
print(f"Checkpoint steps saved to: {output_dir / 'checkpoint_steps.npy'}")

print("\nAll data successfully exported!")
